In [ ]:
# Install required packages in Google Colab
!pip install numpy==1.19.2
!pip install tensorflow==2.4
!pip install torch==1.7.0
!pip install torchsummary==1.5.1
!pip install torchvision==0.8.1
!pip install matplotlib
!pip install ipython

In [ ]:
import os
import time
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
from IPython import display
import matplotlib.pyplot as plt
# %matplotlib inline
from tensorflow import keras

In [ ]:
tf.__version__

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()
x_train = x_train.reshape(x_train.shape[0], 28, 28, 1).astype('float32')
x_train = (x_train - 127.5) / 127.5 # Normalize the images to [-1, 1]
# Batch and shuffle the data
train_dataset = tf.data.Dataset.from_tensor_slices(x_train).\
shuffle(60000).batch(128)

In [ ]:
x_train.shape

In [ ]:
BUFFER_SIZE = 60000
BATCH_SIZE = 128

In [ ]:
latent_dim = 100
image_dim = 784

In [ ]:
num_examples_to_generate = 25
# We will reuse this seed overtime to visualize progress
seed = tf.random.normal([num_examples_to_generate, latent_dim])

In [ ]:
def generator(image_dim):

    inputs = keras.Input(shape=(100,), name='input_layer')
    x = layers.Dense(128, kernel_initializer=tf.keras.initializers.he_uniform, name='dense_1')(inputs)
    #print(x.dtype)
    x = layers.LeakyReLU(0.2, name='leaky_relu_1')(x)
    x = layers.Dense(256, kernel_initializer=tf.keras.initializers.he_uniform, name='dense_2')(x)
    x = layers.BatchNormalization(momentum=0.1,  epsilon=0.8, name='bn_1')(x)
    x = layers.LeakyReLU(0.2, name='leaky_relu_2')(x)
    x = layers.Dense(512, kernel_initializer=tf.keras.initializers.he_uniform, name='dense_3')(x)
    x = layers.BatchNormalization(momentum=0.1,  epsilon=0.8, name='bn_2')(x)
    x = layers.LeakyReLU(0.2, name='leaky_relu_3')(x)
    x = layers.Dense(1024, kernel_initializer=tf.keras.initializers.he_uniform,  name='dense_4')(x)
    x = layers.BatchNormalization(momentum=0.1,  epsilon=0.8, name='bn_3')(x)
    x = layers.LeakyReLU(0.2, name='leaky_relu_4')(x)
    x = layers.Dense(image_dim, kernel_initializer=tf.keras.initializers.he_uniform, activation='tanh',  name='dense_5')(x)
    outputs = layers.Reshape((28, 28, 1), name='Reshape_Layer')(x) # Changed tf.reshape to layers.Reshape
    model = tf.keras.Model(inputs, outputs, name="Generator")
    return model

In [ ]:
generator = generator(image_dim)

In [ ]:
generator.summary()

In [ ]:
def discriminator():
    inputs = keras.Input(shape=(28,28,1), name='input_layer')

    # 卷積層 1: 捕捉基礎紋理
    x = layers.Conv2D(64, (3, 3), strides=(2, 2), padding='same')(inputs)
    x = layers.LeakyReLU(0.2)(x)
    x = layers.Dropout(0.3)(x)

    # 卷積層 2: 捕捉複雜形狀
    x = layers.Conv2D(128, (3, 3), strides=(2, 2), padding='same')(x)
    x = layers.LeakyReLU(0.2)(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Flatten()(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = tf.keras.Model(inputs, outputs, name="Discriminator")
    return model

In [ ]:
discriminator = discriminator()
#discriminator.save('disc.h5')

In [ ]:
discriminator.summary()

In [ ]:
binary_cross_entropy = tf.keras.losses.BinaryCrossentropy()

In [ ]:
def generator_loss(fake_output):
    gen_loss = binary_cross_entropy(tf.ones_like(fake_output), fake_output)
    #print(gen_loss)
    return gen_loss

In [ ]:
def discriminator_loss(real_output, fake_output):
    real_loss = binary_cross_entropy(tf.ones_like(real_output) * 0.9, real_output)
    fake_loss = binary_cross_entropy(tf.zeros_like(fake_output), fake_output)
    total_loss = real_loss + fake_loss
    #print(total_loss)
    return total_loss

In [ ]:
learning_rate = 0.0002

In [ ]:
generator_optimizer = tf.keras.optimizers.Adam(learning_rate = 0.0001, beta_1 = 0.5, beta_2 = 0.999 )
discriminator_optimizer = tf.keras.optimizers.Adam(learning_rate = 0.0008, beta_1 = 0.5, beta_2 = 0.999 )

In [ ]:
@tf.function
def train_step(images):
    noise = tf.random.normal([BATCH_SIZE, latent_dim])

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        generated_images = generator(noise, training=True)

        real_output = discriminator(images, training=True)
        fake_output = discriminator(generated_images, training=True)

        gen_loss = generator_loss(fake_output)
        disc_loss = discriminator_loss(real_output, fake_output)

    gradients_of_gen = gen_tape.gradient(gen_loss, generator.trainable_variables)
    gradients_of_disc = disc_tape.gradient(disc_loss, discriminator.trainable_variables)

    generator_optimizer.apply_gradients(zip(gradients_of_gen, generator.trainable_variables))
    discriminator_optimizer.apply_gradients(zip(gradients_of_disc, discriminator.trainable_variables))

    # 回傳除了 loss 之外的分數
    return gen_loss, disc_loss, tf.reduce_mean(real_output), tf.reduce_mean(fake_output)


In [ ]:
!mkdir tensor

In [ ]:
import os
checkpoint_dir = './training_checkpoints'
checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt")
checkpoint = tf.train.Checkpoint(generator_optimizer=generator_optimizer,
                                 discriminator_optimizer=discriminator_optimizer,
                                 generator=generator,
                                 discriminator=discriminator)

In [ ]:
def train(dataset, epochs):
    G_history = []
    D_history = []

    for epoch in range(epochs):
        start = time.time()
        G_loss_list, D_loss_list = [], []
        real_score_list, fake_score_list = [], []

        for image_batch in dataset:
            gen_loss, disc_loss, real_score, fake_score = train_step(image_batch)
            G_loss_list.append(gen_loss)
            D_loss_list.append(disc_loss)
            real_score_list.append(real_score)
            fake_score_list.append(fake_score)

        G_epoch_loss = np.mean([g.numpy() for g in G_loss_list])
        D_epoch_loss = np.mean([d.numpy() for d in D_loss_list])
        real_score_epoch = np.mean([r.numpy() for r in real_score_list])
        fake_score_epoch = np.mean([f.numpy() for f in fake_score_list])

        G_history.append(G_epoch_loss)
        D_history.append(D_epoch_loss)

        display.clear_output(wait=True)
        print(f"Epoch {epoch+1}")
        print(f"G loss: {G_epoch_loss:.4f}, D loss: {D_epoch_loss:.4f}")
        print(f"Discriminator scores -> Real: {real_score_epoch:.4f}, Fake: {fake_score_epoch:.4f}")

        generate_and_save_images(generator, epoch + 1, seed)

        if (epoch + 1) % 15 == 0:
            checkpoint.save(file_prefix=checkpoint_prefix)

        print('Time for epoch {} is {} sec'.format(epoch + 1, time.time()-start))

    plt.plot(G_history, label='Generator')
    plt.plot(D_history, label='Discriminator')
    plt.legend()
    plt.title("Training Loss")
    plt.savefig("tensor/loss.png")
    plt.show()
    plt.close()

    generate_and_save_images(generator, epochs, seed)

In [ ]:
def generate_and_save_images(model, epoch, test_input):
  # Notice `training` is set to False.
  # This is so all layers run in inference mode (batchnorm).
    predictions = model(test_input, training=False)
    #print(predictions.shape)
    fig = plt.figure(figsize=(4,4))

    for i in range(predictions.shape[0]):
        plt.subplot(5, 5, i+1)
        pred = (predictions[i, :, :, 0] + 1) * 127.5
        pred = np.array(pred)
        plt.imshow(pred.astype(np.uint8), cmap='gray')
        plt.axis('off')

    plt.savefig('tensor/image_at_epoch_{:d}.png'.format(epoch))
    plt.show()

In [ ]:
train(train_dataset, 200)